# IndoBERT Production Training for Indonesian Toxic Speech

This notebook trains `indolem/indobertweet-base-uncased` for binary Indonesian toxic speech classification.

Production flow:

1. Load and validate `dataset/indonesian_toxicspeech.csv`.
2. Split train/validation/test with stratification.
3. Fine-tune IndoBERTweet.
4. Tune the toxic-class decision threshold on validation data.
5. Report final test metrics once with the selected threshold.
6. Export Hugging Face and FP32 ONNX CPU inference artifacts.
7. Run CPU inference smoke tests with ONNX Runtime.


## Setup

Run this cell in Colab before training. It upgrades only notebook-specific libraries and leaves Colab-managed core packages at the runtime versions to avoid dependency conflicts.

`onnxscript` is required by PyTorch's Dynamo ONNX exporter. The ONNX path stays FP32 because the int8 candidate showed unacceptable probability drift.


In [ ]:
%pip install -q --upgrade --upgrade-strategy only-if-needed transformers datasets accelerate onnx onnxscript onnxruntime scikit-learn


In [ ]:
import importlib.metadata as importlib_metadata
import inspect
import json
import math
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "indolem/indobertweet-base-uncased"
MODEL_KEY = "indobertweet_production"
LABEL_MAPPING = {0: "non_toxic", 1: "toxic"}
ID2LABEL = {idx: label for idx, label in LABEL_MAPPING.items()}
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}

ARTIFACT_DIR = Path("artifacts")
OUTPUT_DIR = Path("outputs")
PT_DIR = ARTIFACT_DIR / "toxic_speech_model_pt"
ONNX_DIR = ARTIFACT_DIR / "toxic_speech_model_onnx_cpu"

for path in [ARTIFACT_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
CUDA_DEVICE_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "cpu"
IS_A100 = bool(USE_CUDA and "A100" in CUDA_DEVICE_NAME.upper())
if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
PACKAGE_DISTRIBUTIONS = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "onnxscript": "onnxscript",
    "scikit-learn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
}


def installed_package_versions():
    versions = {}
    for display_name, distribution_name in PACKAGE_DISTRIBUTIONS.items():
        try:
            versions[display_name] = importlib_metadata.version(distribution_name)
        except importlib_metadata.PackageNotFoundError:
            versions[display_name] = "not installed"
    return versions


print(f"Seed set to {SEED}")
for package_name, package_version in installed_package_versions().items():
    print(f"{package_name}: {package_version}")
print(f"Torch device: {DEVICE}")
if USE_CUDA:
    print(f"CUDA device: {CUDA_DEVICE_NAME}")
    print(f"A100 optimized training profile: {IS_A100}")


## Dataset Loading and Validation

Input contract: a CSV with exactly `text` and `is_toxic`, where `is_toxic` is binary (`0 = non_toxic`, `1 = toxic`). By default, the notebook loads the dataset directly from the project GitHub raw URL. Text is only stripped for surrounding whitespace; slang, punctuation, informal words, and toxic terms are preserved because they can carry classification signal.


In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/BayuSatrio2804/Indonesia-Toxic-Speech-Detector/refs/heads/main/dataset/indonesian_toxicspeech.csv"

# Set DATASET_PATH manually if you want to use a local/uploaded CSV instead of the GitHub raw dataset.
DATASET_PATH = None
DEFAULT_DATASET_PATHS = [
    Path("dataset/indonesian_toxicspeech.csv"),
    Path("../dataset/indonesian_toxicspeech.csv"),
    Path("/content/indonesian_toxicspeech.csv"),
    Path("/content/drive/MyDrive/indonesian_toxicspeech.csv"),
]

if DATASET_PATH is not None:
    data_source = DATASET_PATH
elif DATASET_URL:
    data_source = DATASET_URL
else:
    data_source = next((candidate for candidate in DEFAULT_DATASET_PATHS if candidate.exists()), None)

if data_source is None:
    raise FileNotFoundError(
        "Dataset not found. Set DATASET_URL or DATASET_PATH to a CSV with text and is_toxic columns."
    )

raw_df = pd.read_csv(data_source)
print(f"Loaded {data_source} with shape {raw_df.shape}")
raw_df.head()


In [ ]:
EXPECTED_COLUMNS = {"text", "is_toxic"}
actual_columns = set(raw_df.columns)
if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(f"Expected columns {EXPECTED_COLUMNS}, got {actual_columns}")

df = raw_df.copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].copy()
df["is_toxic"] = df["is_toxic"].astype(int)

labels = set(df["is_toxic"].unique().tolist())
if not labels.issubset({0, 1}):
    raise ValueError(f"Labels must be binary 0/1, got {sorted(labels)}")

word_lengths = df["text"].str.split().str.len()
print(f"Rows after validation: {len(df):,}")
print("Class distribution:")
print(df["is_toxic"].value_counts().sort_index().rename(index=LABEL_MAPPING))
print("Missing values:")
print(df.isna().sum())
print(f"Duplicate text count: {df['text'].duplicated().sum():,}")
print("Word length summary:")
print(word_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="is_toxic", ax=axes[0])
axes[0].set_title("Class Distribution")
axes[0].set_xticklabels([LABEL_MAPPING[int(tick.get_text())] for tick in axes[0].get_xticklabels()])
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Rows")

sns.histplot(word_lengths, bins=40, ax=axes[1])
axes[1].set_title("Text Length Distribution")
axes[1].set_xlabel("Words")
plt.tight_layout()


## Split

The split is fixed at `80/10/10` with stratification and seed `42`. Validation is used for threshold tuning. Test is touched once for final reporting after the threshold is selected.


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["is_toxic"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["is_toxic"],
)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    distribution = frame["is_toxic"].value_counts(normalize=True).sort_index().to_dict()
    print(f"{name}: {len(frame):,} rows, class ratios={distribution}")


## Metrics Helpers


In [ ]:
def extract_logits(outputs):
    if hasattr(outputs, "logits"):
        outputs = outputs.logits
    if isinstance(outputs, (tuple, list)):
        outputs = outputs[0]
    if hasattr(outputs, "detach"):
        outputs = outputs.detach().cpu().numpy()
    logits = np.asarray(outputs)
    if logits.ndim != 2 or logits.shape[1] < 2:
        raise ValueError(f"Expected logits with shape (n_examples, n_labels), got {logits.shape}")
    return logits


def softmax_np(logits):
    logits = extract_logits(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def binary_metrics(y_true, toxic_probs, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    toxic_probs = np.asarray(toxic_probs)
    y_pred = (toxic_probs >= threshold).astype(int)
    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    metrics["roc_auc"] = float(roc_auc_score(y_true, toxic_probs)) if len(np.unique(y_true)) == 2 else None
    return metrics


def print_metrics(title, metrics):
    print(title)
    for key in ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc"]:
        value = metrics.get(key)
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
    print(f"  confusion_matrix: {metrics['confusion_matrix']}")


## IndoBERTweet Tokenization and Training

`classifier.weight` and `classifier.bias` are expected to be newly initialized when `AutoModelForSequenceClassification` loads this base checkpoint. The `cls.predictions.*` keys are the masked-language-model head from the pretraining checkpoint and are expected to be unused by sequence classification. The training cell below is the downstream training step that makes the new classification head usable for inference.

The `Trainer` call uses the current `processing_class=tokenizer` API instead of the deprecated `tokenizer=` argument, and warmup is configured with `warmup_steps` instead of deprecated `warmup_ratio`. On NVIDIA A100, the training profile uses larger batches, bf16, TF32, fused AdamW, grouped lengths, and dataloader workers to better use the GPU.


In [ ]:
MAX_LENGTH = int(min(256, max(64, math.ceil(word_lengths.quantile(0.95) / 16) * 16)))
MAX_LENGTH = max(MAX_LENGTH, 128)
print(f"Selected max sequence length: {MAX_LENGTH}")


def make_hf_dataset(frame):
    return Dataset.from_pandas(
        frame[["text", "is_toxic"]].rename(columns={"is_toxic": "labels"}).reset_index(drop=True),
        preserve_index=False,
    )


def align_model_special_tokens(model, tokenizer):
    for attr in ["pad_token_id", "bos_token_id", "eos_token_id"]:
        if hasattr(tokenizer, attr):
            token_id = getattr(tokenizer, attr)
            setattr(model.config, attr, token_id)
            if getattr(model, "generation_config", None) is not None:
                setattr(model.generation_config, attr, token_id)
    return model


def load_tokenizer_and_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    added_tokens = 0
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        added_tokens += 1

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    if added_tokens:
        model.resize_token_embeddings(len(tokenizer))
    return align_model_special_tokens(model, tokenizer), tokenizer


def tokenize_datasets(tokenizer, train_frame, val_frame, test_frame, max_length=128):
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    train_ds = make_hf_dataset(train_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    val_ds = make_hf_dataset(val_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    test_ds = make_hf_dataset(test_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    return train_ds, val_ds, test_ds


def compute_trainer_metrics(eval_pred):
    logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]
    probs = softmax_np(logits)[:, 1]
    return {key: value for key, value in binary_metrics(labels, probs, threshold=0.5).items() if key != "confusion_matrix"}


def make_training_profile(use_cuda, is_a100):
    if is_a100:
        return {
            "name": "a100_bf16_large_batch",
            "train_batch_size": 64,
            "eval_batch_size": 128,
            "dataloader_num_workers": 2,
            "group_by_length": True,
            "use_fused_adamw": True,
            "num_train_epochs": 5,
            "early_stopping_patience": 3,
            "label_smoothing_factor": 0.05,
        }
    if use_cuda:
        return {
            "name": "cuda_conservative",
            "train_batch_size": 16,
            "eval_batch_size": 32,
            "dataloader_num_workers": 2,
            "group_by_length": True,
            "use_fused_adamw": True,
            "num_train_epochs": 5,
            "early_stopping_patience": 3,
            "label_smoothing_factor": 0.05,
        }
    return {
        "name": "cpu",
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "dataloader_num_workers": 0,
        "group_by_length": False,
        "use_fused_adamw": False,
        "num_train_epochs": 3,
        "early_stopping_patience": 2,
        "label_smoothing_factor": 0.0,
    }


TRAINING_PROFILE = make_training_profile(USE_CUDA, IS_A100)
print(f"Training profile: {TRAINING_PROFILE}")


def make_training_args(output_dir, use_cuda, train_size):
    params = inspect.signature(TrainingArguments).parameters
    bf16_enabled = bool(use_cuda and torch.cuda.is_bf16_supported())
    fp16_enabled = bool(use_cuda and not bf16_enabled)
    train_batch_size = TRAINING_PROFILE["train_batch_size"]
    num_train_epochs = TRAINING_PROFILE["num_train_epochs"]
    steps_per_epoch = math.ceil(train_size / train_batch_size)
    warmup_steps = max(1, round(steps_per_epoch * num_train_epochs * 0.06))
    common = {
        "output_dir": str(output_dir),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": train_batch_size,
        "per_device_eval_batch_size": TRAINING_PROFILE["eval_batch_size"],
        "num_train_epochs": num_train_epochs,
        "weight_decay": 0.01,
        "label_smoothing_factor": TRAINING_PROFILE["label_smoothing_factor"],
        "warmup_steps": warmup_steps,
        "logging_steps": 50,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
    }
    dataloader_num_workers = TRAINING_PROFILE["dataloader_num_workers"]
    if "eval_strategy" in params:
        common["eval_strategy"] = "epoch"
    else:
        common["evaluation_strategy"] = "epoch"
    if "bf16" in params:
        common["bf16"] = bf16_enabled
    if "fp16" in params:
        common["fp16"] = fp16_enabled
    if "tf32" in params:
        common["tf32"] = bool(use_cuda)
    if "optim" in params and TRAINING_PROFILE["use_fused_adamw"]:
        common["optim"] = "adamw_torch_fused"
    if "dataloader_num_workers" in params:
        common["dataloader_num_workers"] = dataloader_num_workers
    if "dataloader_pin_memory" in params:
        common["dataloader_pin_memory"] = bool(use_cuda)
    if "dataloader_persistent_workers" in params:
        common["dataloader_persistent_workers"] = dataloader_num_workers > 0
    if "dataloader_prefetch_factor" in params and dataloader_num_workers > 0:
        common["dataloader_prefetch_factor"] = 2
    if "group_by_length" in params:
        common["group_by_length"] = TRAINING_PROFILE["group_by_length"]
    if "save_safetensors" in params:
        common["save_safetensors"] = True
    return TrainingArguments(**common)


def make_data_collator(tokenizer):
    pad_to_multiple_of = 8 if USE_CUDA else None
    return DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=pad_to_multiple_of)


def make_trainer(model, tokenizer, train_ds, val_ds):
    return Trainer(
        model=model,
        args=make_training_args(OUTPUT_DIR / MODEL_KEY, USE_CUDA, len(train_ds)),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=make_data_collator(tokenizer),
        compute_metrics=compute_trainer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=TRAINING_PROFILE["early_stopping_patience"])],
    )


def predict_transformer_probs(trainer, dataset):
    predictions = trainer.predict(dataset)
    return softmax_np(predictions.predictions)[:, 1]


In [ ]:
model, tokenizer = load_tokenizer_and_model(MODEL_NAME)
train_ds, val_ds, test_ds = tokenize_datasets(tokenizer, train_df, val_df, test_df, max_length=MAX_LENGTH)
trainer = make_trainer(model, tokenizer, train_ds, val_ds)

print(f"Training {MODEL_NAME}")
start = time.perf_counter()
trainer.train()
train_seconds = time.perf_counter() - start
print(f"Training finished in {train_seconds:.1f} seconds")


## Validation Threshold Tuning and Final Test Metrics

The decision threshold is selected on validation data by maximizing toxic-class F1. The selected threshold is then applied to the test split once for final reporting.


In [ ]:
val_probs = predict_transformer_probs(trainer, val_ds)
test_probs = predict_transformer_probs(trainer, test_ds)

metrics_at_05_val = binary_metrics(val_df["is_toxic"], val_probs, threshold=0.5)
metrics_at_05_test = binary_metrics(test_df["is_toxic"], test_probs, threshold=0.5)
print_metrics("Validation metrics at threshold 0.5", metrics_at_05_val)
print_metrics("Test metrics at threshold 0.5", metrics_at_05_test)


In [ ]:
def tune_threshold(y_true, toxic_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
    rows = []
    for threshold in thresholds:
        metrics = binary_metrics(y_true, toxic_probs, threshold=float(threshold))
        rows.append({"threshold": float(threshold), **metrics})
    table = pd.DataFrame(rows)
    best_row = table.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).iloc[0]
    return float(best_row["threshold"]), table

selected_threshold, threshold_table = tune_threshold(val_df["is_toxic"], val_probs)
validation_metrics = binary_metrics(val_df["is_toxic"], val_probs, threshold=selected_threshold)
final_test_metrics = binary_metrics(test_df["is_toxic"], test_probs, threshold=selected_threshold)

print(f"Selected threshold: {selected_threshold:.2f}")
print_metrics("Validation metrics at selected threshold", validation_metrics)
print_metrics("Final test metrics at selected threshold", final_test_metrics)
print("\nClassification report on final test split:")
print(classification_report(
    test_df["is_toxic"],
    (test_probs >= selected_threshold).astype(int),
    target_names=[LABEL_MAPPING[0], LABEL_MAPPING[1]],
    zero_division=0,
))

threshold_table.sort_values("f1", ascending=False).head(10)


## Export Production Artifacts

Artifacts are written under `artifacts/`, and the Colab download zip contains only the files needed for CPU inference:

- `toxic_speech_model_pt/`: Hugging Face model and tokenizer.
- `toxic_speech_model_onnx_cpu/model.onnx`: FP32 ONNX export for ONNX Runtime CPU inference.
- `toxic_speech_model_onnx_cpu/`: tokenizer and model config used by the backend.
- `label_mapping.json`, `threshold.json`, `metrics.json`: backend metadata included in the inference zip.

The notebook uses PyTorch's Dynamo ONNX exporter with opset 18, fixed sequence length (`MAX_LENGTH`), and dynamic batch size. It does not run ONNX quantization because the int8 candidate drift was too large for deployment.


In [ ]:
for target in [PT_DIR, ONNX_DIR]:
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(PT_DIR))
tokenizer.save_pretrained(str(PT_DIR))
print(f"Saved Hugging Face model and tokenizer to {PT_DIR}")


In [ ]:
class SequenceClassificationOnnxWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        return self.base_model(**kwargs).logits


try:
    import onnxscript  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch's Dynamo ONNX exporter requires onnxscript. "
        "Run the setup cell, then restart the runtime if needed."
    ) from exc

ONNX_MODEL_PATH = ONNX_DIR / "model.onnx"
OPSET_VERSION = 18
base_export_model = trainer.accelerator.unwrap_model(trainer.model) if hasattr(trainer, "accelerator") else trainer.model
export_model = SequenceClassificationOnnxWrapper(base_export_model.to("cpu").float()).eval()
example_texts = test_df["text"].head(2).tolist() if len(test_df) >= 2 else ["contoh teks biasa", "contoh teks toxic"]
encoded_example = tokenizer(
    example_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

input_names = ["input_ids", "attention_mask"]
example_inputs = [encoded_example["input_ids"], encoded_example["attention_mask"]]
if "token_type_ids" in encoded_example:
    input_names.append("token_type_ids")
    example_inputs.append(encoded_example["token_type_ids"])

output_names = ["logits"]
batch_dim = torch.export.Dim("batch", min=1, max=64)
dynamic_shapes = tuple({0: batch_dim} for _ in input_names)

with torch.no_grad(), warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=".*cache_position.*deprecated.*",
        category=FutureWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message=".*isinstance\\(treespec, LeafSpec\\).*deprecated.*",
        category=FutureWarning,
    )
    onnx_program = torch.onnx.export(
        export_model,
        tuple(example_inputs),
        f=None,
        input_names=input_names,
        output_names=output_names,
        dynamic_shapes=dynamic_shapes,
        opset_version=OPSET_VERSION,
        dynamo=True,
        optimize=True,
        verify=True,
        external_data=False,
    )
onnx_program.save(str(ONNX_MODEL_PATH))
export_method = "torch.onnx.export dynamo=True fixed_sequence_dynamic_batch"

print(f"Exported ONNX model to {ONNX_MODEL_PATH}")
print(f"ONNX export method: {export_method}")
print(f"ONNX inputs: {input_names}")


In [ ]:
import onnx
import onnxruntime as ort

ONNX_PROB_DIFF_TOLERANCE = 0.10
onnx_export_status = "fp32_onnx_cpu"

onnx.checker.check_model(str(ONNX_MODEL_PATH))


def onnx_toxic_probs(model_path, texts):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    input_names = [item.name for item in session.get_inputs()]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in input_names if name in encoded}
    logits = session.run(None, feed)[0]
    return softmax_np(logits)[:, 1]


def pytorch_toxic_probs(texts):
    pt_model = trainer.model.to("cpu").float().eval()
    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    with torch.no_grad():
        logits = pt_model(**encoded).logits.numpy()
    return softmax_np(logits)[:, 1]


export_comparison_texts = test_df["text"].head(8).tolist() if len(test_df) else [
    "Aku suka belajar machine learning hari ini.",
    "Dasar bodoh sekali kelakuan kamu.",
]
pt_export_probs = pytorch_toxic_probs(export_comparison_texts)
onnx_export_probs = onnx_toxic_probs(ONNX_MODEL_PATH, export_comparison_texts)
onnx_export_max_prob_diff = float(np.abs(pt_export_probs - onnx_export_probs).max())
onnx_export_status = f"fp32_onnx_cpu: max_probability_difference={onnx_export_max_prob_diff:.6f}"

print(f"ONNX checker passed: {ONNX_MODEL_PATH}")
print(f"FP32 ONNX max probability difference: {onnx_export_max_prob_diff:.4f}")
if onnx_export_max_prob_diff > ONNX_PROB_DIFF_TOLERANCE:
    print("Warning: FP32 ONNX probabilities differ more than expected. Inspect export settings before deployment.")

tokenizer.save_pretrained(str(ONNX_DIR))
trainer.model.config.save_pretrained(str(ONNX_DIR))
print(f"Saved ONNX CPU inference files to {ONNX_DIR}")


In [ ]:
def directory_size_mb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total_bytes = sum(item.stat().st_size for item in path.rglob("*") if item.is_file())
    return total_bytes / (1024 ** 2)

label_mapping_payload = {"0": "non_toxic", "1": "toxic", "label2id": LABEL2ID}
threshold_payload = {
    "threshold": float(selected_threshold),
    "objective": "maximize toxic-class F1 on validation split",
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
}
metrics_payload = {
    "seed": SEED,
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "train_seconds": float(train_seconds),
    "training_profile": TRAINING_PROFILE,
    "device": {"name": CUDA_DEVICE_NAME, "is_cuda": USE_CUDA, "is_a100": IS_A100},
    "split_rows": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "validation_metrics_at_05": metrics_at_05_val,
    "test_metrics_at_05": metrics_at_05_test,
    "selected_threshold": float(selected_threshold),
    "validation_metrics": validation_metrics,
    "test_metrics": final_test_metrics,
    "onnx_export": {
        "method": export_method,
        "opset_version": OPSET_VERSION,
        "status": onnx_export_status,
        "max_probability_difference": onnx_export_max_prob_diff,
    },
    "package_versions": installed_package_versions(),
    "artifact_sizes_mb": {
        "pytorch": directory_size_mb(PT_DIR),
        "onnx_cpu": directory_size_mb(ONNX_DIR),
    },
}

(ARTIFACT_DIR / "label_mapping.json").write_text(json.dumps(label_mapping_payload, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "threshold.json").write_text(json.dumps(threshold_payload, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

INFERENCE_BUNDLE_DIR = OUTPUT_DIR / "toxic_speech_inference_artifacts"
if INFERENCE_BUNDLE_DIR.exists():
    shutil.rmtree(INFERENCE_BUNDLE_DIR)
INFERENCE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(ONNX_DIR, INFERENCE_BUNDLE_DIR / ONNX_DIR.name)
for metadata_file in ["label_mapping.json", "threshold.json", "metrics.json"]:
    shutil.copy2(ARTIFACT_DIR / metadata_file, INFERENCE_BUNDLE_DIR / metadata_file)

zip_path = shutil.make_archive(str(INFERENCE_BUNDLE_DIR), "zip", INFERENCE_BUNDLE_DIR)
print(f"Wrote metadata files under {ARTIFACT_DIR}")
print(f"Created CPU inference archive: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print(f"Colab download helper unavailable or skipped: {exc}")


## Production Inference Helper

The prediction contract is:

```python
{"label": "toxic" | "non_toxic", "score": toxic_probability, "threshold": selected_threshold}
```


In [ ]:
import onnxruntime as ort

with open(ARTIFACT_DIR / "threshold.json", "r", encoding="utf-8") as handle:
    threshold_config = json.load(handle)
INFERENCE_THRESHOLD = float(threshold_config["threshold"])

ort_tokenizer = AutoTokenizer.from_pretrained(str(ONNX_DIR), use_fast=True)
ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), providers=["CPUExecutionProvider"])
ort_input_names = [item.name for item in ort_session.get_inputs()]
print(f"Loaded ONNX Runtime CPU model: {ONNX_MODEL_PATH}")
print(f"ORT inputs: {ort_input_names}")


def predict_toxic(text: str) -> dict:
    encoded = ort_tokenizer(
        str(text).strip(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in ort_input_names if name in encoded}
    logits = ort_session.run(None, feed)[0]
    toxic_score = float(softmax_np(logits)[0, 1])
    label_id = int(toxic_score >= INFERENCE_THRESHOLD)
    return {
        "label": LABEL_MAPPING[label_id],
        "score": toxic_score,
        "threshold": INFERENCE_THRESHOLD,
    }


def predict_toxic_batch(texts):
    clean_texts = [str(text).strip() for text in texts]
    encoded = ort_tokenizer(
        clean_texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in ort_input_names if name in encoded}
    logits = ort_session.run(None, feed)[0]
    toxic_scores = softmax_np(logits)[:, 1]
    return [
        {"label": LABEL_MAPPING[int(score >= INFERENCE_THRESHOLD)], "score": float(score), "threshold": INFERENCE_THRESHOLD}
        for score in toxic_scores
    ]


In [ ]:
smoke_examples = [
    "Aku suka belajar machine learning hari ini.",
    "Tolong jangan menghina orang lain di komentar.",
    "Dasar bodoh sekali kelakuan kamu.",
    "Diskusinya panas tapi tetap saling menghormati.",
    "Komentar seperti itu tidak pantas dan menyakitkan.",
]

for text, prediction in zip(smoke_examples, predict_toxic_batch(smoke_examples)):
    print({"text": text, **prediction})


In [ ]:
pt_cpu_tokenizer = AutoTokenizer.from_pretrained(str(PT_DIR), use_fast=True)
pt_cpu_model = AutoModelForSequenceClassification.from_pretrained(str(PT_DIR)).to("cpu").eval()
print(f"Loaded PyTorch CPU model: {PT_DIR}")
print(f"Loaded ONNX Runtime CPU model: {ONNX_MODEL_PATH}")
print(f"ONNX artifact status: {onnx_export_status}")


def predict_toxic_pytorch(text: str) -> dict:
    return predict_toxic_pytorch_batch([text])[0]


def predict_toxic_pytorch_batch(texts):
    clean_texts = [str(text).strip() for text in texts]
    encoded = pt_cpu_tokenizer(
        clean_texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    with torch.inference_mode():
        logits = pt_cpu_model(**encoded).logits
    toxic_scores = softmax_np(logits)[:, 1]
    return [
        {"label": LABEL_MAPPING[int(score >= INFERENCE_THRESHOLD)], "score": float(score), "threshold": INFERENCE_THRESHOLD}
        for score in toxic_scores
    ]


def benchmark_callable(fn, *args, runs=50):
    start = time.perf_counter()
    for _ in range(runs):
        fn(*args)
    return (time.perf_counter() - start) * 1000 / runs


latency_texts = test_df["text"].head(16).tolist() if len(test_df) else smoke_examples
single_text = latency_texts[0]

for _ in range(3):
    predict_toxic_pytorch(single_text)
    predict_toxic_pytorch_batch(latency_texts)
    predict_toxic(single_text)
    predict_toxic_batch(latency_texts)

runs = 50
batch_runs = 20
benchmark_rows = []
for engine_name, artifact_path, single_fn, batch_fn in [
    ("PyTorch CPU", str(PT_DIR), predict_toxic_pytorch, predict_toxic_pytorch_batch),
    ("ONNX Runtime CPU", str(ONNX_MODEL_PATH), predict_toxic, predict_toxic_batch),
]:
    single_ms = benchmark_callable(single_fn, single_text, runs=runs)
    batch_ms = benchmark_callable(batch_fn, latency_texts, runs=batch_runs)
    benchmark_rows.append({
        "engine": engine_name,
        "artifact": artifact_path,
        "single_text_ms": single_ms,
        "batch_ms": batch_ms,
        "batch_size": len(latency_texts),
        "batch_per_text_ms": batch_ms / len(latency_texts),
    })

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df)
for row in benchmark_rows:
    print(
        f"{row['engine']}: single={row['single_text_ms']:.2f} ms, "
        f"batch({row['batch_size']})={row['batch_ms']:.2f} ms, "
        f"batch_per_text={row['batch_per_text_ms']:.2f} ms"
    )


In [ ]:
comparison_texts = test_df["text"].head(8).tolist() if len(test_df) else smoke_examples

pt_probs = pytorch_toxic_probs(comparison_texts)
onnx_probs = np.array([row["score"] for row in predict_toxic_batch(comparison_texts)])
abs_diff = np.abs(pt_probs - onnx_probs)
comparison_df = pd.DataFrame({
    "text": comparison_texts,
    "pytorch_toxic_prob": pt_probs,
    "onnx_cpu_toxic_prob": onnx_probs,
    "absolute_difference": abs_diff,
})
display(comparison_df)
print(f"Max absolute probability difference: {abs_diff.max():.4f}")
if abs_diff.max() > ONNX_PROB_DIFF_TOLERANCE:
    print("Warning: ONNX CPU predictions differ more than expected. Inspect export settings before deployment.")


## Minimal Backend Loading Flow

Expected files after unzipping `toxic_speech_inference_artifacts.zip`:

- `toxic_speech_model_onnx_cpu/model.onnx`
- `toxic_speech_model_onnx_cpu/tokenizer.json` and tokenizer config files
- `threshold.json`
- `label_mapping.json`
- `metrics.json`

Backend inference flow:

1. Load tokenizer with `AutoTokenizer.from_pretrained("toxic_speech_model_onnx_cpu")`.
2. Load `model.onnx` with `onnxruntime.InferenceSession(..., providers=["CPUExecutionProvider"])`.
3. Tokenize request text with truncation, padding, and the saved `MAX_LENGTH` from `metrics.json`.
4. Run ONNX logits, apply softmax, and read toxic-class probability.
5. Compare the toxic probability against `threshold.json`.
6. Return `label`, `score`, and `threshold`.
